# 06 · LSTM 核心实现

目标：理解 `i,f,g,o` 四门、cell state 的加法记忆路径，以及 `dh/dc` 双路径 BPTT。

In [ ]:
import inspect, os
from pathlib import Path
if not (Path.cwd() / 'common').exists() and (Path.cwd().parent / 'common').exists():
    os.chdir(Path.cwd().parent)
import matplotlib.pyplot as plt
import numpy as np
import torch
from common.gradcheck import check_gradient, numerical_gradient
from lstm.numpy_impl import sigmoid, init_parameters, step_forward, sequence_forward, sequence_backward
from lstm.torch_impl import ManualLSTMCell, ManualRecurrentLayer
from lstm.experiments import set_forget_bias

## 1. 四门与两条状态路径

一次仿射产生 `[B,4H]`，按 `i,f,g,o` 切分：

$$i=sigmoid(a_i); f=sigmoid(a_f); g=tanh(a_g); o=sigmoid(a_o)$$
$$c_t=f*c_{t-1}+i*g; h_t=o*tanh(c_t)$$

`c_t` 负责长期保存，`h_t` 是暴露给当前输出和下一步的视图。

In [ ]:
def notebook_lstm_step(x_t, state, params):
    h_prev, c_prev = state
    gates = (x_t @ params['weight_ih'].T + params['bias_ih']
             + h_prev @ params['weight_hh'].T + params['bias_hh'])
    ai, af, ag, ao = np.split(gates, 4, axis=1)
    i, f, g, o = sigmoid(ai), sigmoid(af), np.tanh(ag), sigmoid(ao)
    c_t = f * c_prev + i * g
    h_t = o * np.tanh(c_t)
    return h_t, c_t

rng = np.random.default_rng(42)
example_params = init_parameters(3, 4, rng)
x_np = rng.normal(size=(2, 3))
state_np = (rng.normal(size=(2, 4)), rng.normal(size=(2, 4)))
expected_h, expected_c, cache = step_forward(x_np, state_np, example_params)
actual_h, actual_c = notebook_lstm_step(x_np, state_np, example_params)
np.testing.assert_allclose(actual_h, expected_h)
np.testing.assert_allclose(actual_c, expected_c)
print('h/c:', actual_h.shape, actual_c.shape, '| cache:', sorted(cache))

## 2. 双路径 BPTT

反向时先合并隐藏梯度，再通过 $h_t=o*tanh(c_t)$ 向 cell path 增加一项：

$$dc_{total}=dc_{future}+dh_t*o*(1-tanh^2(c_t))$$

随后它分别流向旧 cell、forget gate、input gate 和 candidate。

In [ ]:
print(inspect.getsource(sequence_forward))
print(inspect.getsource(sequence_backward))

In [ ]:
rng = np.random.default_rng(7)
x = rng.normal(size=(2, 2, 2)).astype(np.float64)
h0 = rng.normal(size=(2, 2)).astype(np.float64)
c0 = rng.normal(size=(2, 2)).astype(np.float64)
params = init_parameters(2, 2, rng)
outputs, _, caches = sequence_forward(x, (h0, c0), params)
doutputs = rng.normal(size=outputs.shape)
dx, _, _ = sequence_backward(doutputs, caches)
def objective():
    current, _, _ = sequence_forward(x, (h0, c0), params)
    return float(np.sum(current * doutputs))
result = check_gradient('lstm-dx', dx, numerical_gradient(objective, x))
print(result)
assert result.passed

## 3. PyTorch 对齐与 forget bias

下面先对齐官方 Cell，再观察 forget bias 改变时最终 loss 对早期输入的梯度。

In [ ]:
print(inspect.getsource(ManualLSTMCell.forward))
manual = ManualLSTMCell(3, 4).double()
official = torch.nn.LSTMCell(3, 4).double()
official.load_state_dict(manual.state_dict())
x_t = torch.randn(2, 3, dtype=torch.float64)
state = (torch.randn(2, 4, dtype=torch.float64), torch.randn(2, 4, dtype=torch.float64))
manual_state, official_state = manual(x_t, state), official(x_t, state)
torch.testing.assert_close(manual_state[0], official_state[0])
torch.testing.assert_close(manual_state[1], official_state[1])
print('官方 LSTMCell 前向对齐通过')

In [ ]:
torch.manual_seed(42)
curves = {}
for forget_bias in (0.0, 1.0, 2.0):
    layer = ManualRecurrentLayer(3, 8).double()
    set_forget_bias(layer.cell, forget_bias)
    sequence = torch.randn(1, 40, 3, dtype=torch.float64, requires_grad=True)
    hidden, _ = layer(sequence)
    hidden[:, -1].square().sum().backward()
    curves[forget_bias] = sequence.grad.norm(dim=(0, 2)).detach().numpy()
for label, curve in curves.items():
    plt.semilogy(curve + 1e-20, label=f'forget bias={label}')
plt.xlabel('time index'); plt.ylabel('||dL/dx_t||'); plt.legend(); plt.title('LSTM gradient flow');
plt.show()

## 4. 回忆区

不要查看上面的实现，凭记忆补出四门和两条状态更新。

In [ ]:
RUN_RECALL_TESTS = False
def recall_lstm_step(x_t, state, params):
    # TODO: 返回 h_t, c_t
    pass
if RUN_RECALL_TESTS:
    recalled_h, recalled_c = recall_lstm_step(x_np, state_np, example_params)
    expected_h, expected_c = notebook_lstm_step(x_np, state_np, example_params)
    np.testing.assert_allclose(recalled_h, expected_h)
    np.testing.assert_allclose(recalled_c, expected_c)
    print('回忆测试通过')

### 复盘问题

1. `dc_total` 的两个来源是什么？
2. 哪条路径不必经过 tanh 导数？
3. forget gate 接近 1 是否等于一定记住有用信息？
4. `h` 与 `c` 为什么都要跨 batch detach？